[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/30_sequence_models/04_seq2seq_and_attention.ipynb)

# Bài 04 — Seq2seq và attention

**🎯 Mục tiêu**
- Kiến trúc **encoder–decoder** (seq2seq) cho bài toán chuỗi → chuỗi; **teacher forcing**; giải mã tham lam (greedy), beam search.
- **Nút cổ chai** của vector ngữ cảnh cố định: đo độ chính xác theo độ dài.
- **Attention** (dot-product, Luong): mỗi bước giải mã "nhìn lại" mọi trạng thái encoder; trực quan hoá ma trận attention.
- Giới hạn còn lại (ngoại suy độ dài) và cầu nối tới Transformer (chương 31).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/30_sequence_models

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
    from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
print("thiết bị:", device)

## 1. Bài toán và dữ liệu

Bài toán đồ chơi nhưng đủ để thấy bản chất: **đảo ngược** một dãy chữ số (độ dài 3–15). Dịch máy, tóm tắt, nhận dạng tiếng nói đều có dạng "chuỗi vào → chuỗi ra, độ dài khác nhau".

Token: chữ số 0–9, `PAD` (đệm), `SOS` (bắt đầu giải mã), `EOS` (kết thúc).

In [ ]:
PAD, SOS, EOS, V = 10, 11, 12, 13


def make_batch(n, g, min_len=3, max_len=15):
    lens = torch.randint(min_len, max_len + 1, (n,), generator=g)
    L = int(lens.max())
    src = torch.full((n, L), PAD)
    tgt_in = torch.full((n, L + 1), PAD)
    tgt_out = torch.full((n, L + 1), PAD)
    for i, l in enumerate(lens.tolist()):
        seq = torch.randint(0, 10, (l,), generator=g)
        rev = seq.flip(0)
        src[i, :l] = seq
        tgt_in[i, 0] = SOS                                              # đầu vào decoder: SOS + đích dịch phải 1 bước
        tgt_in[i, 1:l + 1] = rev
        tgt_out[i, :l] = rev                                            # đích: chuỗi đảo + EOS
        tgt_out[i, l] = EOS
    return src, lens, tgt_in, tgt_out


src, lens, ti, to = make_batch(2, torch.Generator().manual_seed(0), 4, 6)
print("nguồn :", src.tolist())
print("dec in:", ti.tolist())
print("đích  :", to.tolist())

## 2. Encoder–decoder

- **Encoder** (GRU) đọc chuỗi nguồn → trạng thái cuối $\mathbf h_{enc}$ (vector ngữ cảnh) và các trạng thái từng bước.
- **Decoder** (GRU) khởi tạo bằng $\mathbf h_{enc}$, sinh từng token. Khi **train**: đầu vào mỗi bước là token **đúng** của bước trước (**teacher forcing**) → train song song, ổn định. Khi **suy luận**: đầu vào là token **mô hình vừa dự đoán** (greedy: lấy argmax).
- `pack_padded_sequence`: encoder dừng đúng ở token thật cuối cùng của mỗi chuỗi (không đọc phần đệm) — quên bước này làm trạng thái cuối khác nhau giữa lúc train (có đệm) và lúc test.

**Attention (Luong, dot-product)**: tại bước giải mã có trạng thái $\mathbf s_t$, tính điểm với **mọi** trạng thái encoder $\mathbf h_j$, chuẩn hoá softmax, lấy trung bình có trọng số:
$$e_{tj} = \mathbf s_t^\top\mathbf h_j, \qquad \alpha_{tj} = \frac{\exp e_{tj}}{\sum_k\exp e_{tk}}, \qquad \mathbf c_t = \sum_j\alpha_{tj}\mathbf h_j, \qquad \hat y_t = \text{softmax}(W[\mathbf s_t; \mathbf c_t])$$

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, attention, emb=32, hidden=128):
        super().__init__()
        self.attention = attention
        self.emb = nn.Embedding(V, emb, padding_idx=PAD)
        self.encoder = nn.GRU(emb, hidden, batch_first=True)
        self.decoder = nn.GRU(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden * (2 if attention else 1), V)

    def encode(self, src, lens):
        packed = pack_padded_sequence(self.emb(src), lens.cpu(), batch_first=True, enforce_sorted=False)
        enc, h = self.encoder(packed)
        enc, _ = pad_packed_sequence(enc, batch_first=True, total_length=src.size(1))
        return enc, h

    def readout(self, dec, enc, mask):
        if not self.attention:
            return self.out(dec), None
        scores = dec @ enc.transpose(1, 2)                              # (N, T_dec, T_enc): s_t · h_j
        scores = scores.masked_fill(~mask[:, None, :], -1e9)             # không chú ý vào PAD
        attn = torch.softmax(scores, dim=-1)
        context = attn @ enc
        return self.out(torch.cat([dec, context], dim=-1)), attn

    def forward(self, src, lens, tgt_in):                               # teacher forcing: cả chuỗi đích một lần
        enc, h = self.encode(src, lens)
        dec, _ = self.decoder(self.emb(tgt_in), h)
        return self.readout(dec, enc, src != PAD)[0]

    @torch.no_grad()
    def greedy(self, src, lens, max_len):
        enc, h = self.encode(src, lens)
        mask = src != PAD
        y = torch.full((src.size(0), 1), SOS, device=src.device)
        outs, attns = [], []
        for _ in range(max_len):
            dec, h = self.decoder(self.emb(y), h)
            logits, attn = self.readout(dec, enc, mask)
            y = logits.argmax(-1)
            outs.append(y)
            attns.append(attn)
        return torch.cat(outs, dim=1), attns

In [ ]:
def train(attention, steps=2000, seed=0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = Seq2Seq(attention).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    for _ in range(steps):
        src, lens, tgt_in, tgt_out = make_batch(128, g)
        logits = model(src.to(device), lens, tgt_in.to(device))
        loss = nn.functional.cross_entropy(logits.reshape(-1, V), tgt_out.to(device).reshape(-1), ignore_index=PAD)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    return model


def exact_match_by_length(model, lengths=(5, 10, 15, 20), seed=99):
    g = torch.Generator().manual_seed(seed)
    result = {}
    for L in lengths:
        src, lens, _, tgt_out = make_batch(500, g, L, L)
        pred, _ = model.greedy(src.to(device), lens, L + 1)
        result[L] = (pred.cpu() == tgt_out).all(dim=1).float().mean().item()
    return result


models = {}
for attention in [False, True]:
    name = "có attention" if attention else "không attention"
    t0 = time.perf_counter()
    models[name] = train(attention)
    acc = exact_match_by_length(models[name])
    print(f"{name:16s}: đúng hoàn toàn theo độ dài {acc} | {time.perf_counter() - t0:.0f}s")

- **Không attention**: chuỗi ngắn gần đúng hết, nhưng độ chính xác **sụt mạnh** khi chuỗi dài hơn — toàn bộ chuỗi nguồn phải nén vào **một** vector 128 chiều (nút cổ chai), và decoder phải "nhớ" token đầu tiên của nguồn (cần sinh **cuối cùng**) qua cả hai mạng.
- **Có attention**: gần như hoàn hảo ở mọi độ dài đã train — decoder truy cập trực tiếp trạng thái encoder cần thiết ở mỗi bước.
- **Độ dài 20** (dài hơn mọi chuỗi lúc train): **cả hai** đều thất bại hoàn toàn — mô hình không tự **ngoại suy** sang độ dài chưa từng thấy (một giới hạn còn tồn tại cả ở Transformer; phần nào do decoder chưa từng "đi" quá 16 bước).

## 3. Mô hình chú ý vào đâu?

In [ ]:
g = torch.Generator().manual_seed(7)
src, lens, _, tgt_out = make_batch(1, g, 10, 10)
pred, attns = models["có attention"].greedy(src.to(device), lens, 11)
A = torch.cat(attns, dim=1)[0].cpu().numpy()                            # (bước giải mã, vị trí nguồn)
print("nguồn :", src[0].tolist())
print("dự đoán:", pred[0].cpu().tolist(), "(12 = EOS)")
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(A, cmap="viridis")
ax.set_xticks(range(10), src[0].tolist())
ax.set_yticks(range(11), [str(t) if t < 10 else "EOS" for t in pred[0].cpu().tolist()])
ax.set_xlabel("token nguồn")
ax.set_ylabel("token sinh ra")
ax.set_title("Trọng số attention")
plt.show()
print("vị trí nguồn được chú ý nhiều nhất ở mỗi bước:", A[:10].argmax(1).tolist(), "| vị trí của token cần sinh:", list(range(9, -1, -1)))

Ma trận attention có dạng **đường chéo phụ**: khi sinh từng token của chuỗi đảo, mô hình chú ý vào đúng vùng tương ứng ở cuối → đầu chuỗi nguồn — nó tự học **căn chỉnh** (alignment) mà không ai dạy. Ở đây đường chéo lệch một vị trí so với "lý tưởng": encoder là GRU **một chiều**, nên trạng thái $\mathbf h_{j+1}$ vẫn chứa thông tin về token $j$ — mô hình có thể đọc token cần sinh từ trạng thái ngay sau nó (encoder hai chiều thường cho căn chỉnh sắc nét hơn). Trong dịch máy, ma trận này cho thấy từ nào ở câu nguồn tương ứng với từ nào ở câu đích (Bahdanau et al., 2015).

## 4. Giải mã: greedy và beam search

- **Greedy**: mỗi bước lấy token xác suất cao nhất — nhanh, nhưng một lựa chọn sai sớm không sửa được.
- **Beam search**: giữ $k$ chuỗi ứng viên tốt nhất theo tổng log-xác suất ở mỗi bước — thường tốt hơn cho dịch máy/tóm tắt; với sinh văn bản tự do, lấy mẫu (nhiệt độ, top-$k$, top-$p$) cho kết quả tự nhiên hơn (chương 31).
- **Exposure bias**: train với teacher forcing nhưng suy luận với dự đoán của chính mình → lỗi tích luỹ; các biện pháp: scheduled sampling, huấn luyện với mục tiêu ở mức chuỗi.

## 5. Từ attention đến Transformer

Attention giải quyết nút cổ chai, nhưng encoder/decoder vẫn là RNN: tuần tự, khó song song, đường đi giữa hai token xa vẫn dài. Ý tưởng của Transformer (Vaswani et al., 2017 — "Attention Is All You Need", chương 31): **bỏ hẳn hồi quy**, dùng **self-attention** — mỗi token chú ý trực tiếp tới mọi token khác trong cùng chuỗi — kèm mã hoá vị trí; huấn luyện song song toàn bộ chuỗi trên GPU.

| | RNN seq2seq | + attention | Transformer |
|---|---|---|---|
| Ngữ cảnh cho decoder | 1 vector | mọi trạng thái encoder | mọi token (self + cross attention) |
| Song song khi train | không (theo thời gian) | không | có |
| Đường đi giữa 2 token cách $n$ | $O(n)$ | $O(n)$ trong encoder | $O(1)$ |
| Chi phí theo độ dài | $O(n)$ | $O(n^2)$ cho attention | $O(n^2)$ |

## 🧠 Kiểm tra nhanh

1. Teacher forcing là gì? Vì sao dùng khi train? Hệ quả khi suy luận?
2. Nút cổ chai của seq2seq không attention là gì? Thí nghiệm cho thấy điều đó thế nào?
3. Viết công thức dot-product attention.
4. Vì sao cần `pack_padded_sequence` (hoặc mask) với chuỗi có độ dài khác nhau?
5. Ma trận attention của bài toán đảo chuỗi có dạng gì? Vì sao?

## 📌 Tóm tắt

- Seq2seq: encoder nén chuỗi nguồn, decoder sinh chuỗi đích; teacher forcing khi train, greedy/beam khi suy luận.
- Một vector ngữ cảnh là nút cổ chai → độ chính xác giảm theo độ dài; attention cho decoder truy cập mọi trạng thái encoder và tự học căn chỉnh.
- Không ngoại suy được độ dài chưa thấy; Transformer thay hồi quy bằng self-attention.

## 📚 Tài liệu tham khảo

- Sutskever, Vinyals & Le (2014) — *Sequence to Sequence Learning with Neural Networks*
- Bahdanau, Cho & Bengio (2015) — *Neural Machine Translation by Jointly Learning to Align and Translate*
- Luong, Pham & Manning (2015) — *Effective Approaches to Attention-based Neural Machine Translation*
- Vaswani et al. (2017) — *Attention Is All You Need*